In [0]:
# =============================================================================
# NOTEBOOK  : gold_fact_customer_sales
# PURPOSE   : silver.pos_transactions + silver.customers
#             → gold.fact_customer_sales
# LAYER     : Gold (enriched transaction fact)
# FREQUENCY : Daily (after silver_pos_transactions + silver_customers complete)
# WRITE     : Dynamic partition overwrite by transaction_date
# GRAIN     : one row per transaction_id
#
# LOGIC: His logic uses two intermediate gold dims (dim_customer_segment_new,
#        dim_pos_transactions) which is redundant — channel and payment_method
#        are already in silver.pos_transactions, customer attributes in
#        silver.customers. We simplify to direct join — same output columns,
#        cleaner pipeline, no intermediate tables needed.
#        transaction_ts replaces his event_timestamp (our column name).
#        transaction_date replaces his inventory_date (our column name).
#        Added: gender, zip_code, preferred_categories, customer_tenure_days.
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import GOLD_FACT_CUSTOMER_SALES_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col, to_date,
    broadcast, datediff, current_date
)
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
POS_TABLE      = cfg["tables"]["silver_pos_transactions"]
CUSTOMER_TABLE = cfg["tables"]["silver_customers"]
TARGET_TABLE   = cfg["tables"]["gold_fact_customer_sales"]
PIPELINE       = "gold_fact_customer_sales"

In [0]:
# ── Read + Join + Write ───────────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, POS_TABLE)
 
try:
    # Incremental read — only new transactions since last run
    last_run_time = get_last_successful_run_time(spark, PROJECT_ROOT, PIPELINE)
 
    if last_run_time:
        pos_df = (
            spark.read.table(POS_TABLE)
            .filter(col("processed_at") > lit(last_run_time))
        )
    else:
        pos_df = spark.read.table(POS_TABLE)
 
    rows_read = pos_df.count()
    print(f"[READ] {rows_read} new POS transactions")
 
    if rows_read == 0:
        print("[SKIP] No new transactions to process")
        end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
                  rows_read=0, rows_written=0)
        raise SystemExit(0)
 
    # Customers — full read, broadcast (100K rows)
    cust_df = (
        spark.read.table(CUSTOMER_TABLE)
        .select("customer_id", "age_group", "gender", "zip_code",
                "loyalty_tier", "preferred_categories", "first_purchase_date")
    )
 
    # Left join — anonymous transactions (null customer_id) are preserved
    df = (
        pos_df.alias("tx")
        .join(broadcast(cust_df).alias("c"), "customer_id", "left")
        .withColumn("customer_tenure_days",
            datediff(current_date(), col("c.first_purchase_date")))
        .withColumn("unit_price",
            col("tx.unit_price").cast("double"))        # Decimal → Double explicitly
        .withColumn("total_amount",
            col("tx.total_amount").cast("double"))      # Decimal → Double explicitly
        .withColumn("transaction_date",
            to_date(col("tx.transaction_ts")))
        .withColumn("_fact_processed_at", current_timestamp())
        .withColumn("pipeline_run_id",    lit(run_id))
        .select([f.name for f in GOLD_FACT_CUSTOMER_SALES_SCHEMA.fields])
    )
 
    rows_written = df.count()
 
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
 
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("transaction_date")
        .option("overwriteSchema", "false")
        .saveAsTable(TARGET_TABLE)
    )
 
    print(f"[DONE] {rows_written} rows written to {TARGET_TABLE}")
 
    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
              rows_read=rows_read, rows_written=rows_written)
 
except SystemExit:
    pass
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col
 
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written").show(truncate=False)
 
print("Row count:", spark.read.table(TARGET_TABLE).count())
spark.read.table(TARGET_TABLE).groupBy("loyalty_tier", "channel").count().show()